# 01 · Pitch increment and exploration parameters
Agents evaluated on the **training wind** (11 m/s, TI = 10 %) against ROSCO.

* Section 5.4 / Table 3: pitch increment $\Delta\beta$.
* Section 5.4: initial exploration rate $\varepsilon_0$ and decay $\varepsilon_{\mathrm{decay}}$ during policy refinement.

Selection index: NEVI = (E / E_ROSCO) / scaled STD of power.
Analysis window: first 2000 s of simulation, discarding the initial 200 s transient (200–2000 s).

In [1]:
import pandas as pd
from utils import load_fast_file, window, scaled_std, generated_energy

T0, T1 = 200, 2000  # analysis window [s]

rosco, _ = load_fast_file("train_11ti10_rosco.out")

# Pitch increment candidates (Table 3)
INCREMENTS = {
    "Δβ = 0.125°": "train_11ti10_ddqn_db0.125.out",
    "Δβ = 0.025°": "train_11ti10_ddqn_db0.025.out",
    "Δβ = 0.010°": "train_11ti10_ddqn_db0.010.out",
    "Δβ = 0.005°": "train_11ti10_ddqn_db0.005.out",
}

# Initial exploration rate, Δβ = 0.010°, ε_decay = 10^-3.25
EPSILON_0 = {
    "ε0 = 0.025": "train_11ti10_ddqn_db0.010_pr800_eps0.025_dec3.25.out",
    "ε0 = 0.050": "train_11ti10_ddqn_db0.010_pr500_eps0.05_dec3.25.out",
    "ε0 = 0.100": "train_11ti10_ddqn_db0.010_pr600_eps0.10_dec3.25.out",
    "ε0 = 0.150": "train_11ti10_ddqn_db0.010_pr500_eps0.15_dec3.25.out",
}

# Exploration decay, Δβ = 0.010°, ε0 = 0.025
EPSILON_DECAY = {
    "ε_decay = 10^-2.75": "train_11ti10_ddqn_db0.010_pr800_eps0.025_dec2.75.out",
    "ε_decay = 10^-3.25": "train_11ti10_ddqn_db0.010_pr800_eps0.025_dec3.25.out",
    "ε_decay = 10^-3.50": "train_11ti10_ddqn_db0.010_pr800_eps0.025_dec3.50.out",
    "ε_decay = 10^-3.75": "train_11ti10_ddqn_db0.010_pr800_eps0.025_dec3.75.out",
}

In [2]:
def nevi_table(files):
    # Energy, power variability and NEVI of each agent relative to ROSCO in [T0, T1].
    # NEVI is NaN when the power is constant in the window (e.g. the turbine produced no power).
    ref = window(rosco, T0, T1)
    e_ref = generated_energy(ref)
    rows = []
    for label, df in [("ROSCO", rosco)] + [(k, load_fast_file(f)[0]) for k, f in files.items()]:
        d = window(df, T0, T1)
        energy = generated_energy(d)
        s_pwr = scaled_std(d["GenPwr"], ref["GenPwr"])
        rows.append({
            "Controller": label,
            "Mean power [kW]": d["GenPwr"].mean(),
            "Energy [kWh]": energy,
            "Energy diff. [%]": 100 * (energy - e_ref) / e_ref,
            "Power STD [kW]": d["GenPwr"].std(),
            "Power scaled STD": s_pwr,
            "TB scaled STD": scaled_std(d["TwrBsMyt"], ref["TwrBsMyt"]),
            "BR scaled STD": scaled_std(d["RootMyb1"], ref["RootMyb1"]),
            "NEVI": (energy / e_ref) / s_pwr if s_pwr > 0 else float("nan"),
            "Last time step [s]": df["Time"].iloc[-1],
            "Covers window": bool(df["Time"].iloc[-1] >= T1),
        })
    return pd.DataFrame(rows).set_index("Controller").round(3)


print(f"Analysis window: {T0}-{T1} s")

Analysis window: 200-2000 s


## Pitch increment (Table 3)

In [3]:
nevi_table(INCREMENTS)

,Mean power [kW],Energy [kWh],Energy diff. [%],Power STD [kW],Power scaled STD,TB scaled STD,BR scaled STD,NEVI,Last time step [s],Covers window
Controller,,,,,,,,,,
ROSCO,3339.410,1669.705,0.000,125.065,0.125,0.139,0.137,8.020,5500.000,True
Δβ = 0.125°,3291.470,1645.735,-1.436,277.187,0.276,0.892,0.492,3.567,2092.250,True
Δβ = 0.025°,3367.587,1683.793,0.844,92.670,0.092,0.307,0.216,10.915,2095.300,True
Δβ = 0.010°,3369.152,1684.576,0.891,80.840,0.081,0.208,0.172,12.518,5500.000,True
Δβ = 0.005°,3392.477,1696.238,1.589,224.788,0.224,0.238,0.191,4.533,2154.375,True


## Initial exploration rate ε0
ε0 = 0.150 produces no power during the whole simulation, so its NEVI is NaN.

In [4]:
nevi_table(EPSILON_0)

,Mean power [kW],Energy [kWh],Energy diff. [%],Power STD [kW],Power scaled STD,TB scaled STD,BR scaled STD,NEVI,Last time step [s],Covers window
Controller,,,,,,,,,,
ROSCO,3339.410,1669.705,0.000,125.065,0.125,0.139,0.137,8.020,5500.00,True
ε0 = 0.025,3369.241,1684.620,0.893,76.891,0.077,0.205,0.171,13.161,5500.00,True
ε0 = 0.050,3374.385,1687.192,1.047,79.878,0.080,0.204,0.171,12.688,5273.15,True
ε0 = 0.100,3369.366,1684.683,0.897,79.139,0.079,0.208,0.173,12.788,5500.00,True
ε0 = 0.150,0.000,0.000,-100.000,0.000,0.000,0.034,0.109,NaN,5500.00,True


## Exploration decay ε_decay
Runs with *Covers window* = False stopped before 2000 s: 10^-2.75 diverges (power above rated) and 10^-3.75 produces no power.

In [5]:
nevi_table(EPSILON_DECAY)

,Mean power [kW],Energy [kWh],Energy diff. [%],Power STD [kW],Power scaled STD,TB scaled STD,BR scaled STD,NEVI,Last time step [s],Covers window
Controller,,,,,,,,,,
ROSCO,3339.410,1669.705,0.000,125.065,0.125,0.139,0.137,8.020,5500.000,True
ε_decay = 10^-2.75,4839.504,1019.089,-38.966,707.606,0.705,0.563,0.452,0.865,958.075,False
ε_decay = 10^-3.25,3369.241,1684.620,0.893,76.891,0.077,0.205,0.171,13.161,5500.000,True
ε_decay = 10^-3.50,3369.533,1684.766,0.902,78.944,0.079,0.206,0.171,12.820,5500.000,True
ε_decay = 10^-3.75,0.000,0.000,-100.000,0.000,0.000,0.033,0.107,0.000,706.575,False
